In [ ]:
import sys
sys.path.insert(0, '/home/phil/Dev/Python/Single Cell/scvelo')
sys.path.insert(0, '/home/phil/Dev/Python/Single Cell/scanpy')
%load_ext autoreload
%autoreload 2

In [ ]:
import tarfile
from tempfile import gettempdir
from urllib.request import urlretrieve
from pathlib import Path
from string import ascii_lowercase

import matplotlib.pyplot as plt
import tikzplotlib
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
from anndata import AnnData
from rpy2.robjects import r, pandas2ri
%matplotlib inline

download data

In [ ]:
tmp_path = Path(gettempdir()) / "dim-red"
tmp_path.mkdir(exist_ok=True)

if not (tar_path := tmp_path / "destiny.tar.gz").is_file():
    urlretrieve("http://bioconductor.org/packages/release/bioc/src/contrib/destiny_3.0.1.tar.gz", tar_path)

if not (rda_path := tmp_path / "guo_norm.rda").is_file():
    with tarfile.open(tar_path) as tf:
        reader = tf.extractfile("destiny/data/guo_norm.rda")
        rda_path.write_bytes(reader.read())

load data

In [ ]:
pandas2ri.activate()

r["load"](str(rda_path))
adata = AnnData(
    r("t(guo_norm@assayData$exprs)"),
    r("guo_norm@phenoData@data"),
    dict(var_names=r("rownames(guo_norm@assayData$exprs)")),
)
#adata.obs["num_cells"] = pd.Categorical(adata.obs["num_cells"])

analyze

In [ ]:
#sc.pp.log1p(adata)
sc.pp.neighbors(adata, 20)
sc.tl.pca(adata, )
sc.tl.diffmap(adata)
sc.tl.umap(adata)

plot

In [ ]:
print(plt.style.available)

In [ ]:
plt.style.use("seaborn-paper")
sc.settings.set_figure_params(fontsize=10)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Palatino', 'TeX Gyre Pagella']
plt.rcParams['lines.linewidth'] = .5

fig = plt.figure(figsize=(6.5, 1.9))
gs = fig.add_gridspec(1, 4)
kws = [
    dict(x="Gata4", y="Nanog"),
    dict(basis="pca"),
    dict(basis="diffmap"),
    dict(basis="umap"),
]
# hack: https://github.com/theislab/scvelo/issues/150
adata.uns['num_cells_colors'] = list(map(plt.cm.colors.to_hex, plt.cm.viridis(np.linspace(0, 1, 6))))
for letter, kw, ax in zip(ascii_lowercase, kws, map(fig.add_subplot, gs)):
    scv.pl.scatter(adata, **kw, color='num_cells', palette='viridis', frameon=True, title="", ax=ax, show=False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(-0.15, 1, letter, transform=ax.transAxes, size=16, weight='bold')

#plt.xlabel("time (s)")
#plt.ylabel("Voltage (mV)")
#plt.title("Simple plot $\\frac{\\alpha}{2}$")
#plt.grid(True)
 
#tikzplotlib.save("dim-red.tikz", flavor="context")
fig.tight_layout()
fig.savefig("dim-red.pdf")